# Qwen2.5-0.5B SFT + DPO Pilot

Run this notebook on Kaggle with the `GPU T4 x2` accelerator. The scripts are the source of truth; this notebook is a convenient command runner.

In [ ]:
!pip install -U -r requirements.txt
!pip install -e .
!nvidia-smi

In [ ]:
!python scripts/make_alpaca_subset.py --size 50 --seed 42 --output_dir data/alpaca_eval

In [ ]:
!python scripts/generate_alpaca.py \
  --model_name_or_path Qwen/Qwen2.5-0.5B \
  --subset_path data/alpaca_eval/alpacaeval_50_seed42.json \
  --output_path outputs/eval/base_alpacaeval_50.json \
  --generator_name qwen2.5-0.5b-base \
  --batch_size 8

In [ ]:
!python scripts/train_sft.py \
  --model_name_or_path Qwen/Qwen2.5-0.5B \
  --output_dir outputs/sft-pilot \
  --max_train_examples 2000 \
  --max_val_examples 200 \
  --per_device_batch_size 2 \
  --gradient_accumulation_steps 16 \
  --eval_steps 50 \
  --save_steps 100

In [ ]:
!python scripts/generate_alpaca.py \
  --model_name_or_path outputs/sft-pilot/best \
  --subset_path data/alpaca_eval/alpacaeval_50_seed42.json \
  --output_path outputs/eval/sft_alpacaeval_50.json \
  --generator_name qwen2.5-0.5b-sft \
  --batch_size 8

In [ ]:
!python scripts/train_dpo.py \
  --model_name_or_path outputs/sft-pilot/best \
  --output_dir outputs/dpo-pilot \
  --max_train_examples 1000 \
  --max_val_examples 200 \
  --per_device_batch_size 1 \
  --gradient_accumulation_steps 32 \
  --eval_steps 25

In [ ]:
!python scripts/generate_alpaca.py \
  --model_name_or_path outputs/dpo-pilot/best \
  --subset_path data/alpaca_eval/alpacaeval_50_seed42.json \
  --output_path outputs/eval/dpo_alpacaeval_50.json \
  --generator_name qwen2.5-0.5b-dpo \
  --batch_size 8

In [ ]:
!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/dpo_alpacaeval_50.json \
  --b_outputs outputs/eval/sft_alpacaeval_50.json \
  --a_name dpo \
  --b_name sft \
  --output_path outputs/eval/dpo_vs_sft_pairs.jsonl